# 08 — Índices de Seca: SPI e SPEI a partir de dados CHIRPS

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/local/08_drought_indices_spi_spei.pt.ipynb)

**Objetivo de aprendizagem**: calcular o Índice de Precipitação Padronizado (SPI) e o Índice de Precipitação-Evapotranspiração Padronizado (SPEI) em múltiplas escalas temporais (1, 3, 6 e 12 meses) a partir de uma série temporal mensal de precipitação. Ao final deste notebook você saberá montar a tabela de entrada que `lcz_climate_compute_spi` e `lcz_climate_compute_spei` esperam, interpretar as classificações de seca resultantes e entender por que o SPEI — ao incorporar a evapotranspiração — é mais sensível a secas agravadas pelo aquecimento urbano do que o SPI sozinho.

## Por que índices de seca padronizados, e por que múltiplas escalas temporais

O SPI (McKee et al. 1993) é o índice de seca meteorológica mais usado no mundo: transforma a precipitação acumulada em um número adimensional, comparável entre climas e regiões, ajustando uma distribuição gama à série histórica de cada local e depois padronizando o resultado para uma distribuição normal padrão (média 0, desvio-padrão 1). Um SPI de -1.5 significa a mesma coisa em Fortaleza, em Berlim ou em Manaus: uma anomalia de precipitação que historicamente ocorre com a mesma raridade relativa, mesmo que os regimes de chuva absolutos sejam completamente diferentes.

A escala temporal (o número de meses acumulados antes de padronizar) importa tanto quanto o índice em si. **Escalas curtas (1-3 meses)** respondem rápido — refletem o teor de umidade do solo e são as mais relevantes para seca agrícola: uma cultura sente falta de chuva em semanas, não em anos. **Escalas longas (6-12 meses ou mais)** suavizam a variabilidade de curto prazo e capturam o balanço hídrico acumulado que enche (ou esvazia) reservatórios, aquíferos e rios — a chamada seca hidrológica. É por isso que `lcz_climate_compute_spi` e `lcz_climate_compute_spei` calculam por padrão quatro escalas simultâneas (1, 3, 6, 12 meses): nenhuma escala isolada conta a história completa de uma seca.

O SPEI (Vicente-Serrano et al. 2010) resolve uma limitação conhecida do SPI: ele ignora completamente a demanda evaporativa da atmosfera. Em um mundo que aquece, uma cidade pode receber a mesma chuva de sempre e ainda assim entrar em déficit hídrico severo porque a evapotranspiração potencial (ETP) subiu junto com a temperatura. O SPEI substitui a precipitação bruta pelo balanço hídrico D = precipitação − ETP antes de padronizar, tornando-o sensível a secas *induzidas por temperatura* que o SPI simplesmente não enxerga. Isso é particularmente relevante no contexto do LCZ4py: o calor urbano (ilha de calor) eleva a demanda evaporativa localmente além do que a paisagem rural ao redor experimenta, então uma cidade pode apresentar SPEI mais severo que SPI mesmo sob chuva normal — um sinal direto de que o clima urbano está amplificando o estresse hídrico local, não apenas refletindo um déficit regional de chuva.

Rodar esses índices sobre uma série de precipitação CHIRPS **recortada ao contorno do mapa LCZ da cidade** (em vez de usar uma estação única ou uma média regional genérica) amarra este notebook ao restante do fluxo LCZ4py: a mesma extensão espacial que usamos para baixar o mapa LCZ (`lcz_get_map`), extrair parâmetros morfológicos e calcular índices espectrais é agora a área sobre a qual quantificamos o risco de seca — todos os produtos do pacote descrevem consistentemente a mesma "cidade".

Ambas as funções seguem a assinatura climática compartilhada do LCZ4py: `df` de entrada e saída são `pandas.DataFrame` (não raster, não polars) — a única exceção "tabular pura" desta série de tutoriais, porque SPI/SPEI operam sobre uma série temporal agregada, não pixel a pixel.

## Instalar o LCZ4py

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git" 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## Passo 1 — Montar a série temporal de entrada a partir do CHIRPS

Nenhum dataset já incluso no repositório (nem o CSV de estações de Berlim, que só tem temperatura do ar) tem precipitação mensal pronta — este é o passo menos "plug-and-play" de toda a série de tutoriais, então vamos construí-lo com cuidado, explicando cada transformação.

1. Baixamos o mapa LCZ de uma cidade pequena (`lcz_get_map`, notebook `01`) para definir a extensão espacial.
2. Usamos `lcz_grid_chirps(x, resolution="monthly", years=[...])` (notebook `general/07`) para baixar precipitação mensal CHIRPS recortada a essa extensão. O retorno é um `LCZGridResult` cujo `.array` tem forma `(n_meses, H, W)` (mm/mês, NaN fora da área válida) e cujo `.bands[i]` é a data ISO de cada mês.
3. Como SPI/SPEI operam sobre **uma série temporal por unidade** (a assinatura espera uma coluna `code_muni` — herdada da origem do pacote irmão `climasus4r`, que processa municípios brasileiros — mas aqui basta um identificador constante para "esta cidade"), colapsamos cada banda mensal para um único número: a média espacial da precipitação sobre todos os pixels válidos da cidade. Isso produz exatamente uma série temporal mensal, que é o formato de entrada esperado.
4. Montamos o `pandas.DataFrame` com as colunas exigidas: `code_muni` (constante), `date` (mensal, tipo datetime) e a coluna de precipitação, cujo nome deve casar com o parâmetro `var`/`rain_var` das funções — usaremos o nome padrão `rainfall_chirps_mm` para não precisar sobrescrever esse argumento.
5. O SPEI, quando `pet_method="thornthwaite"`, também precisa de uma coluna de temperatura média mensal (`temp_var`, padrão `tair_dry_bulb_c`) para derivar a evapotranspiração potencial internamente (ver `_thornthwaite_pet` no código-fonte). Como não temos ERA5 ou uma estação com temperatura mensal de longo prazo já pronta para esta cidade neste notebook, **sintetizamos** uma série de temperatura mensal plausível (um ciclo sazonal senoidal com ruído) apenas para fins didáticos. Em um estudo real, essa coluna viria de dados observados — por exemplo `lcz_grid_era5`/`lcz_grid_era5_global` (variável `t2m`, já abordada no notebook `general/07`) agregado da mesma forma que fizemos com o CHIRPS, ou de uma estação meteorológica local.

In [2]:
from LCZ4py import lcz_get_map, lcz_grid_chirps

map_path = lcz_get_map(city="Vaduz", isave_map=True)
print(map_path)

06:43:08 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


06:43:08 - rasterio._env - WARNING - CPLE_IllegalArg in tmppu1ie3t0.tif: BLOCKXSIZE can only be used with TILED=YES


06:43:08 - LCZ4py._internal._lcz_map_engine - INFO - Saved map to LCZ4r_output/lcz_map.tif


/Users/co2map/.lcz4r_cache/clipped_a8db9faa23e32c3b.tif


O `lcz_get_map` acima baixa (ou reutiliza do cache) o mapa LCZ de Vaduz — escolhida por sua área pequena, o que mantém os downloads do CHIRPS rápidos para este exemplo didático. Em um estudo real, basta trocar `city=` pela cidade de interesse.

In [3]:
import numpy as np
import pandas as pd

years = [2022, 2023]  # 2 anos ~ 24 meses de dados mensais (mantém o download rápido para este exemplo)
chirps = lcz_grid_chirps(map_path, resolution="monthly", years=years, isave=False)

print(chirps.array.shape, len(chirps.bands))
print(chirps.bands[:3], "...", chirps.bands[-3:])

Using cached CHIRPS stack.
Complete: CHIRPS stack with 24 band(s), cropped to the LCZ map.
(24, 120, 131) 24
['2022-01-01', '2022-02-01', '2022-03-01'] ... ['2023-10-01', '2023-11-01', '2023-12-01']


`chirps.array` tem forma `(n_meses, H, W)`; cada `chirps.bands[i]` é a data ISO (primeiro dia do mês) daquela banda. Agora colapsamos cada banda para a média espacial (ignorando NaN) para obter uma série temporal mensal única representando a cidade inteira.

In [4]:
monthly_rain = np.nanmean(chirps.array.reshape(chirps.array.shape[0], -1), axis=1)
dates = pd.to_datetime(chirps.bands)

# Série de temperatura sintética (ciclo sazonal + ruído) — SUBSTITUA por dados
# observados reais (ex.: ERA5 t2m via lcz_grid_era5_global) em um estudo real.
rng = np.random.default_rng(42)
month_num = dates.month.to_numpy()
synthetic_temp = 10 + 8 * np.sin(2 * np.pi * (month_num - 4) / 12) + rng.normal(0, 0.8, len(dates))

df = pd.DataFrame({
    "code_muni": "vaduz",       # identificador constante: uma única "unidade" (a cidade)
    "date": dates,
    "rainfall_chirps_mm": monthly_rain,
    "tair_dry_bulb_c": synthetic_temp,
})
df = df.sort_values("date").reset_index(drop=True)
df.head()

,code_muni,date,rainfall_chirps_mm,tair_dry_bulb_c
0,vaduz,2022-01-01,37.658882,2.243774
1,vaduz,2022-02-01,111.390411,2.239809
2,vaduz,2022-03-01,34.109623,6.600361
3,vaduz,2022-04-01,94.559219,10.752452
4,vaduz,2022-05-01,111.366608,12.439172


Este `df` — com `code_muni`, `date` mensal, a coluna de precipitação e a coluna de temperatura — é exatamente a forma tabular que `lcz_climate_compute_spi` e `lcz_climate_compute_spei` esperam. Note que, com apenas 2 anos (~24 meses) de dados — um recorte pequeno de propósito para manter os downloads do CHIRPS rápidos neste notebook de demonstração — este exemplo é puramente didático: a OMM recomenda pelo menos 30 anos de calibração para um ajuste gama robusto do SPI; usamos `min_n` mais baixo abaixo só para a demonstração rodar (com 24 meses e escala de 12 meses, só há 13 valores de soma móvel possíveis).

## Passo 2 — `lcz_climate_compute_spi`: SPI em múltiplas escalas

Calcula o SPI para cada escala em `scales` (padrão `(1, 3, 6, 12)` meses). Para cada escala *s* e cada unidade (`code_muni`): soma móvel de *s* meses da precipitação, ajuste de uma gama zero-inflacionada ao período de calibração (`ref_start`/`ref_end`, ou todo o período se `None`) via método dos momentos, e padronização para normal-padrão. `min_n` (padrão 24) é o número mínimo de valores não-NA no período de calibração — abaixo disso a função não calcula o índice para aquela unidade/escala. O retorno é o próprio `df` de entrada com colunas adicionadas `spi_{s}mo` (uma por escala).

Classificação padrão do SPI: ≥2.0 extremamente úmido, 1.5–1.99 muito úmido, 1.0–1.49 moderadamente úmido, -0.99..0.99 normal, -1..-1.49 seca moderada (D1), -1.5..-1.99 seca severa (D2), ≤-2.0 seca extrema (D3-D4).

In [5]:
from LCZ4py import lcz_climate_compute_spi

spi_df = lcz_climate_compute_spi(
    df,
    var="rainfall_chirps_mm",
    scales=(1, 3, 6, 12),
    min_n=12,  # reduzido só porque temos poucos anos de exemplo; use >=24 (idealmente 30 anos) em produção
)
spi_df[["date", "rainfall_chirps_mm", "spi_1mo", "spi_3mo", "spi_6mo", "spi_12mo"]].tail(12)

Computing SPI (Standardized Precipitation Index)
Computing 1-month scale -> spi_1mo...
Computing 3-month scale -> spi_3mo...
Computing 6-month scale -> spi_6mo...
Computing 12-month scale -> spi_12mo...
Done: 24 rows, 0 NA(s) in 'spi_1mo'.


,date,rainfall_chirps_mm,spi_1mo,spi_3mo,spi_6mo,spi_12mo
12,2023-01-01,61.398342,-1.160628,-1.001060,-0.385862,-0.980131
13,2023-02-01,54.622860,-1.328720,-1.322701,-1.110639,-1.361164
14,2023-03-01,140.415771,0.212960,-1.158153,-1.224711,-0.655919
15,2023-04-01,160.115967,0.466478,-0.294653,-1.000358,-0.237968
16,2023-05-01,147.522842,0.306988,0.379470,-0.655228,-0.012553
17,2023-06-01,85.517754,-0.652131,-0.006111,-0.873651,-0.563166
18,2023-07-01,226.610779,1.195349,0.456852,0.009596,-0.114808
19,2023-08-01,350.782135,2.252654,1.644239,1.331577,0.895852
20,2023-09-01,83.560089,-0.689328,1.634020,1.097577,0.449725
21,2023-10-01,139.885239,0.205815,1.160299,1.012329,0.550707


Cada coluna `spi_{s}mo` é independente das outras: um mês pode estar em seca severa na escala de 1 mês (chuva quase nula naquele mês específico) e ainda assim próximo do normal na escala de 12 meses (se os meses anteriores compensaram). Ler as quatro escalas lado a lado é o que permite diferenciar um "mês seco" passageiro de uma seca hidrológica estrutural se acumulando ao longo do ano. Valores `NaN` no início da série são esperados — a soma móvel de *s* meses só existe a partir do *s*-ésimo mês, e a calibração da gama exige `min_n` observações válidas.

## Passo 3 — `lcz_climate_compute_spei`: SPEI incorporando evapotranspiração

Mesma mecânica de múltiplas escalas do SPI, mas em vez de padronizar a precipitação bruta, padroniza o balanço hídrico D = precipitação − ETP (soma móvel de *s* meses), usando a posição de plotagem empírica de Hazen (`p = (rank-0.5)/n`) em vez do ajuste gama — válido para qualquer formato de D, incluindo valores negativos (o balanço hídrico pode ser negativo, ao contrário da precipitação).

A ETP (evapotranspiração potencial) pode vir de duas formas, controladas por `pet_method`:
- `pet_method="column"` (padrão): lê uma coluna de ETP já calculada, `pet_var` (padrão `"pet_mm"`) — útil se você já tem ETP de um produto como o ERA5-Land ou um cálculo Penman-Monteith externo.
- `pet_method="thornthwaite"`: deriva a ETP internamente a partir de `temp_var` (temperatura média mensal, padrão `"tair_dry_bulb_c"`) usando o método de Thornthwaite (1948) — um método simples baseado só em temperatura, sem correção de fotoperíodo/latitude (razoável para regiões tropicais onde a duração do dia varia pouco; menos preciso em latitudes altas com verões/invernos de dia muito longo/curto). É esse o caminho que vamos usar aqui, já que só temos a coluna de temperatura sintética.

In [6]:
from LCZ4py import lcz_climate_compute_spei

spei_df = lcz_climate_compute_spei(
    spi_df,  # reaproveita o df que já tem as colunas spi_*
    rain_var="rainfall_chirps_mm",
    pet_method="thornthwaite",
    temp_var="tair_dry_bulb_c",
    scales=(1, 3, 6, 12),
    min_n=12,
)
spei_df[["date", "rainfall_chirps_mm", "tair_dry_bulb_c", "spi_12mo", "spei_12mo"]].tail(12)

Computing SPEI (Standardized Precipitation-Evapotranspiration Index)
Computing 1-month scale -> spei_1mo...
Computing 3-month scale -> spei_3mo...
Computing 6-month scale -> spei_6mo...
Computing 12-month scale -> spei_12mo...
Done: 24 rows, 0 NA(s) in 'spei_1mo'.


,date,rainfall_chirps_mm,tair_dry_bulb_c,spi_12mo,spei_12mo
12,2023-01-01,61.398342,2.052825,-0.980131,-0.869424
13,2023-02-01,54.622860,3.973590,-1.361164,-1.768825
14,2023-03-01,140.415771,6.374007,-0.655919,-0.615141
15,2023-04-01,160.115967,9.312566,-0.237968,-0.194028
16,2023-05-01,147.522842,14.295001,-0.012553,0.194028
17,2023-06-01,85.517754,16.161097,-0.563166,-0.395725
18,2023-07-01,226.610779,18.702760,-0.114808,0.000000
19,2023-08-01,350.782135,16.888263,0.895852,0.869424
20,2023-09-01,83.560089,13.852110,0.449725,0.395725
21,2023-10-01,139.885239,9.455256,0.550707,0.615141


Compare `spi_12mo` com `spei_12mo` no mesmo mês: se `spei_12mo` for consistentemente mais negativo que `spi_12mo`, isso é o sinal característico de uma seca *agravada pela temperatura* — a chuva por si só não seria classificada como tão severa, mas a demanda evaporativa elevada (temperaturas mais altas puxando mais água do solo e da vegetação) empurra o balanço hídrico para um déficit maior. Em cidades quentes — especialmente sob o efeito de ilha de calor urbana que o LCZ4py caracteriza via `lcz_uhi_intensity`/`lcz_get_ucp` — esse gap SPI-vs-SPEI tende a ser mais pronunciado que no entorno rural, porque o excesso de calor local eleva a ETP localmente sem qualquer mudança na chuva regional. É exatamente esse tipo de evidência — seca "invisível" ao SPI mas capturada pelo SPEI — que motiva rodar ambos os índices, e rodá-los sobre a extensão espacial específica derivada do mapa LCZ da cidade, não sobre uma média regional genérica que dilui o sinal urbano.

## Conclusão e fim da série de tutoriais

Este notebook fechou o arco de 8 tutoriais da série `local/`: partimos de séries temporais estratificadas por LCZ (`01`), passamos por anomalias de temperatura (`02`), intensidade de ilha de calor urbana (`03`), interpolação geoestatística via krigagem (`04`) e via aprendizado de máquina (`05`), métricas climáticas temporais como amplitude térmica diurna e graus-dia (`06`), conforto térmico e calor antropogênico (`07`), e terminamos aqui com índices de seca padronizados em múltiplas escalas temporais (`08`).

Somada à série `general/` — aquisição do mapa LCZ, visualização e estatísticas de área, parâmetros morfológicos, sensoriamento remoto (LST via Planetary Computer), índices espectrais, parâmetros de dossel urbano e dados climáticos/ambientais em grade — você agora tem as 15 peças de um fluxo de trabalho completo de clima urbano baseado em LCZ: desde baixar o mapa de classificação da sua cidade até quantificar seca, calor, conforto térmico e variabilidade espacial de temperatura sobre essa mesma extensão geográfica consistente. O próximo passo natural é aplicar esse fluxo à sua própria cidade — trocando `city="Vaduz"` (ou qualquer outra cidade de exemplo usada ao longo da série) pela área de estudo real, e as estações/grades de exemplo pelos seus próprios dados observacionais.

**Anterior**: `07_thermal_comfort_anthropogenic_heat` (conforto térmico e calor antropogênico).
**Próximo**: nenhum — este é o último notebook da série `local/`, e do arco completo de 15 notebooks `general/` + `local/` do LCZ4py.